# Hierarchical ARM V2 - Visualization Notebook

Notebook này dùng để visualization predictions của mô hình HierarchicalARM_V2:
- Stage 1: ROI Means predictions (518 clusters)
- Stage 2: Full voxel predictions (15724 voxels)

Sử dụng 3D plots để hiển thị predictions và ground truth.

In [1]:
import sys
sys.path.append('/home/sowwn/Workspace/ws/2026/I2fMRI')

import torch
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import yaml
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import seaborn as sns

# Import model and data
from ARMNet.hierarchical_v2 import HierarchicalARM_V2
from data.dataset import create_dataloaders

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 1. Load Config và Model

In [2]:
# Load config
config_path = "/home/sowwn/Workspace/ws/2026/I2fMRI/configs/train_hierarchical_v2.yaml"
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print("Config loaded:")
print(f"  Visual dim: {config['model']['visual_dim']}")
print(f"  fMRI dim: {config['model']['fmri_dim']}")
print(f"  Num clusters: {config['model']['num_clusters']}")
print(f"  Stage 2 embed dim: {config['model']['embed_dim_2']}")
print(f"  Stage 2 depth: {config['model']['depth_2']}")

Config loaded:
  Visual dim: 768
  fMRI dim: 15724
  Num clusters: 518
  Stage 2 embed dim: 512
  Stage 2 depth: 6


In [3]:
# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = HierarchicalARM_V2(
    visual_dim=config['model']['visual_dim'],
    num_clusters=config['model']['num_clusters'],
    fmri_dim=config['model']['fmri_dim'],
    hidden_dim_1=config['model']['hidden_dim_1'],
    embed_dim_2=config['model']['embed_dim_2'],
    depth_2=config['model']['depth_2'],
    num_heads_2=config['model']['num_heads_2'],
    num_global_queries=config['model']['num_global_queries'],
    contrastive_dim=config['model']['contrastive_dim'],
    use_mamba=config['model']['use_mamba']
).to(device)

print(f"\nModel initialized with {sum(p.numel() for p in model.parameters())/1e6:.2f}M parameters")

Using device: cuda
Using Mamba for Stage 1 Selection mechanism.


Initializing Stage 2 with Transformer Backbone.

Model initialized with 390.46M parameters


In [4]:
# Load checkpoint
checkpoint_path = "/home/sowwn/Workspace/ws/2026/I2fMRI/checkpoints/hierarchical_v21_subj01/best_model.pth"
checkpoint = torch.load(checkpoint_path, map_location='cuda')

# Check if checkpoint has 'model_state_dict' key or is the state_dict itself
if 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Checkpoint loaded from epoch {checkpoint.get('epoch', 'unknown')}")
    if 'metrics' in checkpoint:
        print(f"Best validation metrics:")
        for key, val in checkpoint.get('metrics', {}).items():
            print(f"  {key}: {val:.4f}")
else:
    model.load_state_dict(checkpoint)
    print(f"Checkpoint loaded (state dict format)")

model.eval()
print("Model set to evaluation mode")

Checkpoint loaded (state dict format)
Model set to evaluation mode


## 2. Load Data

In [5]:
# Create dataloaders
import os

# Convert relative paths in config to absolute paths
base_path = "/home/sowwn/Workspace/ws/2026/I2fMRI"
datalist_path = os.path.join(base_path, config['data']['datalist_path'])
fmri_path = os.path.join(base_path, config['data']['fmri_path'])
train_embeddings_path = os.path.join(base_path, config['data']['train_embeddings_path'])
test_embeddings_path = os.path.join(base_path, config['data']['test_embeddings_path'])
sub_roi_cluster_path = os.path.join(base_path, config['data']['sub_roi_cluster_path'])

print(f"Loading data from:")
print(f"  Datalist: {datalist_path}")
print(f"  fMRI: {fmri_path}")
print(f"  Train embeddings: {train_embeddings_path}")
print(f"  Test embeddings: {test_embeddings_path}")
print(f"  Sub-ROI clusters: {sub_roi_cluster_path}")

train_loader, val_loader = create_dataloaders(
    datalist_path=datalist_path,
    fmri_path=fmri_path,
    train_embeddings_path=train_embeddings_path,
    test_embeddings_path=test_embeddings_path,
    subjects=[config['data']['subject']],
    batch_size=16,  # Smaller batch for visualization
    average_trials_train=config['data']['average_trials_train'],
    average_trials_val=config['data']['average_trials_val'],
    augment_noise_train=False,  # No augmentation for viz
    sub_roi_cluster_path=sub_roi_cluster_path,
    roi_top_k_percent=config['data']['roi_top_k_percent']
)

print(f"\nData loaded successfully:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")

Loading data from:
  Datalist: /home/sowwn/Workspace/ws/2026/I2fMRI/dataset/nsd/metadata/datalist_mindeye2_sub01.json
  fMRI: /home/sowwn/Workspace/ws/2026/I2fMRI/dataset/nsd/subj01/betas_all_subj01_fp32_renorm.hdf5
  Train embeddings: /home/sowwn/Workspace/ws/2026/I2fMRI/dataset/nsd/embeddings_mindeye2/dinov2_train_sub01.npy
  Test embeddings: /home/sowwn/Workspace/ws/2026/I2fMRI/dataset/nsd/embeddings_mindeye2/dinov2_test_sub01.npy
  Sub-ROI clusters: /home/sowwn/Workspace/ws/2026/I2fMRI/data/sub_roi_labels.npy
Loading datalist from /home/sowwn/Workspace/ws/2026/I2fMRI/dataset/nsd/metadata/datalist_mindeye2_sub01.json...
Loading train embeddings from /home/sowwn/Workspace/ws/2026/I2fMRI/dataset/nsd/embeddings_mindeye2/dinov2_train_sub01.npy...
Loading fMRI data from /home/sowwn/Workspace/ws/2026/I2fMRI/dataset/nsd/subj01/betas_all_subj01_fp32_renorm.hdf5...



Processing Subject 1 for train split...
  Train fMRI: (27000, 15724), 9000 unique images
Loading sub-ROI clusters from /home/sowwn/Workspace/ws/2026/I2fMRI/data/sub_roi_labels.npy...
ROI aggregation: 518 clusters using top-75% (computed on-the-fly)
NeuroFluxDataset: 27000 samples, 9000 unique images, embeddings shape: (9000, 768), without augmentation, with 518 sub-ROIs (top-75% on-the-fly)
Loading datalist from /home/sowwn/Workspace/ws/2026/I2fMRI/dataset/nsd/metadata/datalist_mindeye2_sub01.json...
Loading test embeddings from /home/sowwn/Workspace/ws/2026/I2fMRI/dataset/nsd/embeddings_mindeye2/dinov2_test_sub01.npy...
Loading fMRI data from /home/sowwn/Workspace/ws/2026/I2fMRI/dataset/nsd/subj01/betas_all_subj01_fp32_renorm.hdf5...

Processing Subject 1 for test split...
  Averaging trials for test split...
  Test fMRI: (982, 15724), 982 unique images
Loading sub-ROI clusters from /home/sowwn/Workspace/ws/2026/I2fMRI/data/sub_roi_labels.npy...
ROI aggregation: 518 clusters using to

In [26]:
emb = val_loader.dataset[0]['embedding'].unsqueeze(0)
print(f"Embeddings shape: {emb.shape}")

Embeddings shape: torch.Size([1, 768])


## 3. Run Inference và Thu thập Predictions